<a href="https://colab.research.google.com/github/rperezen/intro-to-ML/blob/main/Homework_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

from google.colab import files
uploaded = files.upload()

df = pd.read_csv(io.BytesIO(uploaded['D3.csv']))
print(df.head())

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['figure.dpi'] = 110
np.set_printoptions(precision=4, suppress=True)

# Load the dataset (no header issues: the file already has X1,X2,X3,Y as headers)
df = pd.read_csv('D3.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
# Quick look at how each variable relates to Y
print(df.corr()['Y'])

## 1. Gradient descent implementation



In [ ]:
def add_bias(X):
    """Prepend a column of ones so theta_0 acts as the intercept."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.hstack([np.ones((X.shape[0], 1)), X])


def compute_cost(X, y, theta):
    """J(theta) = (1/2m) * sum((X@theta - y)^2)"""
    err = X @ theta - y
    return float(err @ err) / (2 * len(y))


def gradient_descent(X, y, alpha, num_iters=2000, tol=1e-9):
    """Batch gradient descent, theta initialised to zeros.

    Returns
    -------
    theta      : learned parameters
    cost_hist  : J(theta) after each iteration
    conv_iter  : first iteration where the cost changed by less than `tol`
                 (our practical definition of "converged"), or None
    """
    m, n = X.shape
    theta = np.zeros(n)                 # <-- required initialisation
    cost_hist = np.zeros(num_iters)
    conv_iter = None
    prev_cost = compute_cost(X, y, theta)

    for i in range(num_iters):
        gradient = (1.0 / m) * (X.T @ (X @ theta - y))
        theta = theta - alpha * gradient        # simultaneous update
        cost_hist[i] = compute_cost(X, y, theta)

        if conv_iter is None and abs(prev_cost - cost_hist[i]) < tol:
            conv_iter = i + 1
        prev_cost = cost_hist[i]

    return theta, cost_hist, conv_iter

In [ ]:
y = df['Y'].values
m = len(y)
LEARNING_RATES = [0.1, 0.05, 0.01]   # explored range: 0.1 down to 0.01
NUM_ITERS = 2000
print(f'm = {m} training examples')

---
# Problem 1 — One explanatory variable at a time

Three separate trainings: x1 x2 x3


In [ ]:
results_p1 = {}

for var in ['X1', 'X2', 'X3']:
    X = add_bias(df[var].values)
    results_p1[var] = {}
    print(f'===== Training on {var} only =====')
    for alpha in LEARNING_RATES:
        theta, hist, conv = gradient_descent(X, y, alpha, NUM_ITERS)
        results_p1[var][alpha] = (theta, hist, conv)
        conv_txt = conv if conv is not None else f'> {NUM_ITERS}'
        print(f'  alpha={alpha:<5} theta0={theta[0]: .4f}  theta1={theta[1]: .4f}  '
              f'final J={hist[-1]:.6f}  iters to converge={conv_txt}')
    print()

In [ ]:
# Summary table
rows = []
for var in ['X1', 'X2', 'X3']:
    for alpha in LEARNING_RATES:
        theta, hist, conv = results_p1[var][alpha]
        rows.append({'Variable': var, 'alpha': alpha,
                     'theta_0': round(theta[0], 4), 'theta_1': round(theta[1], 4),
                     'Final cost J': round(hist[-1], 6),
                     'Iters to converge': conv if conv is not None else f'>{NUM_ITERS}'})
summary_p1 = pd.DataFrame(rows)
summary_p1

### 1.1 The linear model found each explanatory variable

( α = 0.1)

In [ ]:
BEST_ALPHA = 0.1
for var in ['X1', 'X2', 'X3']:
    t = results_p1[var][BEST_ALPHA][0]
    J = results_p1[var][BEST_ALPHA][1][-1]
    sign = '+' if t[1] >= 0 else '-'
    print(f'{var}:  Y = {t[0]:.4f} {sign} {abs(t[1]):.4f} * {var}      (final cost J = {J:.4f})')

### 1.2 final regression model anf loss over iteration per each explanatory variable

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, var in zip(axes, ['X1', 'X2', 'X3']):
    x = df[var].values
    theta = results_p1[var][BEST_ALPHA][0]
    ax.scatter(x, y, s=18, alpha=0.65, label='data')
    xs = np.linspace(x.min(), x.max(), 100)
    sign = '+' if theta[1] >= 0 else '-'
    ax.plot(xs, theta[0] + theta[1] * xs, 'r-', lw=2,
            label=f'Y = {theta[0]:.3f} {sign} {abs(theta[1]):.3f}*{var}')
    ax.set_xlabel(var); ax.set_ylabel('Y')
    ax.set_title(f'Regression on {var}  (J = {results_p1[var][BEST_ALPHA][1][-1]:.3f})')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('p1_regression_lines.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 Loss over iterations per explanatory variable (all learning rates)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, var in zip(axes, ['X1', 'X2', 'X3']):
    for alpha in LEARNING_RATES:
        hist = results_p1[var][alpha][1]
        ax.plot(hist, lw=1.8, label=f'alpha = {alpha}')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Cost  J(theta)')
    ax.set_title(f'Loss vs iteration — {var}')
    ax.set_xlim(0, 1000)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('p1_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Same curves on one axis (alpha = 0.1) to compare the three variables directly
plt.figure()
for var in ['X1', 'X2', 'X3']:
    plt.plot(results_p1[var][BEST_ALPHA][1], lw=2, label=f'{var}  (final J = {results_p1[var][BEST_ALPHA][1][-1]:.3f})')
plt.xlabel('Iteration'); plt.ylabel('Cost  J(theta)')
plt.title('Loss comparison of the three single-variable models (alpha = 0.1)')
plt.xlim(0, 600); plt.legend(); plt.grid(alpha=0.3)
plt.savefig('p1_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Which explanatory variable explains Y best?

In [ ]:
final = {v: results_p1[v][BEST_ALPHA][1][-1] for v in ['X1', 'X2', 'X3']}
for v, j in final.items():
    print(f'{v}: final cost = {j:.6f}')
best = min(final, key=final.get)
print(f'\n--> {best} has the LOWEST loss, so it explains Y best.')
print(f'    Its cost is about {final["X2"]/final["X1"]:.1f}x lower than X2 and '
      f'{final["X3"]/final["X1"]:.1f}x lower than X3.')

In [ ]:
baseline = compute_cost(np.ones((m, 1)), y, np.array([y.mean()]))
print(f'Baseline cost of always predicting mean(Y): {baseline:.4f}')
for v, j in final.items():
    print(f'{v}: R^2 = {1 - j/baseline:.4f}')

---
# Problem 2 — All three explanatory variables together

In [ ]:
X_all = add_bias(df[['X1', 'X2', 'X3']].values)

results_p2 = {}
for alpha in LEARNING_RATES:
    theta, hist, conv = gradient_descent(X_all, y, alpha, NUM_ITERS)
    results_p2[alpha] = (theta, hist, conv)
    conv_txt = conv if conv is not None else f'> {NUM_ITERS}'
    print(f'alpha={alpha:<5} theta={theta}  final J={hist[-1]:.6f}  '
          f'iters to converge={conv_txt}')

In [ ]:
rows = []
for alpha in LEARNING_RATES:
    theta, hist, conv = results_p2[alpha]
    rows.append({'alpha': alpha, 'theta_0': round(theta[0], 4), 'theta_1': round(theta[1], 4),
                 'theta_2': round(theta[2], 4), 'theta_3': round(theta[3], 4),
                 'Final cost J': round(hist[-1], 6),
                 'Iters to converge': conv if conv is not None else f'>{NUM_ITERS}'})
pd.DataFrame(rows)

### 2.1 The best final linear model

In [ ]:
theta_best, hist_best, conv_best = results_p2[BEST_ALPHA]
print('Best run: alpha = 0.1, 2000 iterations, theta initialised to zeros\n')
print(f'Y = {theta_best[0]:.4f} '
      f'{"+" if theta_best[1] >= 0 else "-"} {abs(theta_best[1]):.4f}*X1 '
      f'{"+" if theta_best[2] >= 0 else "-"} {abs(theta_best[2]):.4f}*X2 '
      f'{"+" if theta_best[3] >= 0 else "-"} {abs(theta_best[3]):.4f}*X3')
print(f'\nFinal cost J = {hist_best[-1]:.6f}')
print(f'R^2          = {1 - hist_best[-1]/baseline:.4f}')

In [ ]:
# Independent sanity check against the closed-form (normal equation) solution.
# This is NOT used for training - it only verifies gradient descent converged correctly.
theta_closed = np.linalg.lstsq(X_all, y, rcond=None)[0]
print('Gradient descent :', theta_best)
print('Closed form      :', theta_closed)
print('Max abs difference:', np.abs(theta_best - theta_closed).max())

### 2.2 Loss over iterations

In [ ]:
plt.figure()
for alpha in LEARNING_RATES:
    hist = results_p2[alpha][1]
    plt.plot(hist, lw=1.8, label=f'alpha = {alpha}  (final J = {hist[-1]:.4f})')
plt.xlabel('Iteration'); plt.ylabel('Cost  J(theta)')
plt.title('Problem 2 — loss vs iteration, all three variables')
plt.xlim(0, 2000); plt.legend(); plt.grid(alpha=0.3)
plt.savefig('p2_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Log scale makes the different convergence speeds much clearer
plt.figure()
for alpha in LEARNING_RATES:
    plt.semilogy(results_p2[alpha][1], lw=1.8, label=f'alpha = {alpha}')
plt.xlabel('Iteration'); plt.ylabel('Cost  J(theta)  (log scale)')
plt.title('Problem 2 — loss vs iteration (log scale)')
plt.legend(); plt.grid(alpha=0.3, which='both')
plt.savefig('p2_loss_curves_log.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Effect of the learning rate



In [ ]:
for alpha in [0.01, 0.05, 0.1, 0.12, 0.15, 0.2]:
    theta, hist, conv = gradient_descent(X_all, y, alpha, 500)
    status = 'stable' if np.isfinite(hist[-1]) and hist[-1] < hist[0] else 'DIVERGED'
    print(f'alpha={alpha:<5} final J after 500 iters = {hist[-1]:.6g}   ({status})')

# Theoretical stability limit: alpha < 2 / lambda_max of (1/m) X^T X
lam_max = np.linalg.eigvalsh((X_all.T @ X_all) / m).max()
print(f'\nLargest eigenvalue of (1/m)X^T X = {lam_max:.4f}')
print(f'Theoretical stability limit alpha < 2/lambda_max = {2/lam_max:.4f}')

### 2.4 Predictions for new values of (X1, X2, X3)

In [ ]:
new_points = np.array([[1, 1, 1],
                       [2, 0, 4],
                       [3, 2, 1]], dtype=float)

preds = add_bias(new_points) @ theta_best

for p, yhat in zip(new_points, preds):
    print(f'(X1, X2, X3) = ({p[0]:.0f}, {p[1]:.0f}, {p[2]:.0f})  ->  predicted Y = {yhat:.4f}')

pd.DataFrame({'X1': new_points[:, 0], 'X2': new_points[:, 1],
              'X3': new_points[:, 2], 'Predicted Y': preds.round(4)})

In [ ]:
# Show the arithmetic explicitly for the first point
t = theta_best
p = new_points[0]
print(f'Y = {t[0]:.4f} + ({t[1]:.4f})({p[0]:.0f}) + ({t[2]:.4f})({p[1]:.0f}) + ({t[3]:.4f})({p[2]:.0f})')
print(f'  = {t[0]:.4f} + {t[1]*p[0]:.4f} + {t[2]*p[1]:.4f} + {t[3]*p[2]:.4f}')
print(f'  = {t @ np.array([1, *p]):.4f}')